# NB02: Data Transformation

In [3]:
import pandas as pd
import numpy as np

## Section 1: Creating New "Cleaner" CSV for MSOA Travel Durations

In [4]:
df = pd.read_csv("data/raw/sample_postcodes_durations.csv")
newdf = df[["msoa11", "duration_to_central", "oslaua"]]
newdf.head(1)

,msoa11,duration_to_central,oslaua
0,E02000001,13,E09000001


In [5]:
# Adding Names of the Local Authority Districts

borough_names = pd.read_csv("data/raw/Local_Authority_Districts_Names_and_Codes_UK.csv")
borough_names.head(2)

,LAD25CD,LAD25NM,LAD25NMW,ObjectId
0,E06000001,Hartlepool,NaN,1
1,E06000002,Middlesbrough,NaN,2


In [6]:
borough_names = borough_names[["LAD25CD", "LAD25NM"]]
borough_names.head()

,LAD25CD,LAD25NM
0,E06000001,Hartlepool
1,E06000002,Middlesbrough
2,E06000003,Redcar and Cleveland
3,E06000004,Stockton-on-Tees
4,E06000005,Darlington


In [7]:
merged_df = newdf.merge(borough_names, left_on='oslaua', right_on='LAD25CD', how='left')
merged_df.head(5)

,msoa11,duration_to_central,oslaua,LAD25CD,LAD25NM
0,E02000001,13,E09000001,E09000001,City of London
1,E02000002,72,E09000002,E09000002,Barking and Dagenham
2,E02000003,75,E09000002,E09000002,Barking and Dagenham
3,E02000004,72,E09000002,E09000002,Barking and Dagenham
4,E02000005,72,E09000002,E09000002,Barking and Dagenham


In [8]:
merged_df = merged_df.drop("LAD25CD", axis=1)
merged_df.head()

,msoa11,duration_to_central,oslaua,LAD25NM
0,E02000001,13,E09000001,City of London
1,E02000002,72,E09000002,Barking and Dagenham
2,E02000003,75,E09000002,Barking and Dagenham
3,E02000004,72,E09000002,Barking and Dagenham
4,E02000005,72,E09000002,Barking and Dagenham


In [9]:
# Sanity Check - Do we have all and only 33 local authorities (32 boroughs + City of London) listed?

print(len(merged_df["LAD25NM"].unique()) == 33)

True


In [10]:
# Renaming the borough names heading for ease of use
merged_df.rename(columns={"LAD25NM": "borough"}, inplace=True)

In [ ]:
merged_df.to_csv("data/processed/msoa_duration.csv", index=False)

**Personal Reflection Notes:**

In this section, I "cleaned up" the raw CSV I made in NB01 based on the postcode directory and durations of travel time between each MSOA sample postcode and central London. This involved removing most of the columns from the CSV, and only keeping the important ones that I will need for my NB03 Exploratory Data Analysis. This also involved me adding the borough names for each sample postcode, as this will allow me to more easily conduct borough-level aggregations and analyses of journey times in my NB03 EDA. The postcode directory we used doesn't contain borough names, only their codes stored under the "oslaua" column, therefore I downloaded a CSV of what borough each code corresponds to [from the ONS's Open Geography Portal](https://geoportal.statistics.gov.uk/datasets/5779a9578f0e48ccacef6af41546b56b_0/explore) and uploaded it to my Nuvolos so I could use it to add borough names. I added the borough names using the pd.merge tool that we covered in the Week 7 lecture and lab to merge the 2 dataframes, eventually leaving me with a new CSV containing all the information I need and the borough names, ready to be analysed in NB03.

## Section 2: Creating New "Cleaner" CSVs for Most and Least Deprived Areas

### Section 2.1: Most Deprived Areas

In [12]:
most_df = pd.read_csv("data/raw/most_deprived_postcodes.csv")
most_df.columns

Index(['pcds', 'dointr', 'doterm', 'oscty', 'ced', 'oslaua', 'osward',
       'parish', 'usertype', 'oseast1m', 'osnrth1m', 'osgrdind', 'oshlthau',
       'nhser', 'ctry', 'rgn', 'streg', 'pcon', 'eer', 'teclec', 'ttwa', 'pct',
       'itl', 'statsward', 'oa01', 'casward', 'park', 'lsoa01', 'msoa01',
       'ur01ind', 'oac01', 'oa11', 'lsoa11', 'msoa11', 'wz11', 'ccg', 'bua11',
       'buasd11', 'ru11ind', 'oac11', 'lat', 'long', 'lep1', 'lep2', 'pfa',
       'imd', 'calncv', 'stp', 'duration_to_central', 'total_walking'],
      dtype='object')

In [13]:
most_df = most_df[["pcds", "oslaua", "lsoa11", "imd", "lat", "long", "duration_to_central", "total_walking"]]
most_df.head()

,pcds,oslaua,lsoa11,imd,lat,long,duration_to_central,total_walking
0,E1 7AA,E09000001,E01000005,8678,51.515567,-0.075635,30,20
1,IG11 0AG,E09000002,E01000093,2669,51.531241,0.106421,65,32
2,NW11 9EH,E09000003,E01000221,2878,51.573205,-0.211092,49,23
3,DA8 2AB,E09000004,E01000429,3591,51.478252,0.182770,71,21
4,NW10 0AB,E09000005,E01000601,1192,51.553022,-0.253283,54,26


In [14]:
most_merged = most_df.merge(borough_names, left_on='oslaua', right_on='LAD25CD', how='left')
print(len(most_merged["LAD25NM"].unique()) == 33)

True


In [15]:
most_merged = most_merged.drop("LAD25CD", axis=1)
most_merged.rename(columns={"LAD25NM": "borough"}, inplace=True)
most_merged.head()

,pcds,oslaua,lsoa11,imd,lat,long,duration_to_central,total_walking,borough
0,E1 7AA,E09000001,E01000005,8678,51.515567,-0.075635,30,20,City of London
1,IG11 0AG,E09000002,E01000093,2669,51.531241,0.106421,65,32,Barking and Dagenham
2,NW11 9EH,E09000003,E01000221,2878,51.573205,-0.211092,49,23,Barnet
3,DA8 2AB,E09000004,E01000429,3591,51.478252,0.182770,71,21,Bexley
4,NW10 0AB,E09000005,E01000601,1192,51.553022,-0.253283,54,26,Brent


In [16]:
def haversine_np(lon1, lat1, lon2, lat2):
    """
    Calculate the great circle distance between two points
    on the earth (specified in decimal degrees)
    
    All args must be of equal length.    
    
    """
    lon1, lat1, lon2, lat2 = map(np.radians, [lon1, lat1, lon2, lat2])
    
    dlon = lon2 - lon1
    dlat = lat2 - lat1
    
    a = np.sin(dlat/2.0)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon/2.0)**2
    
    c = 2 * np.arcsin(np.sqrt(a))
    km = 6378.137 * c
    return km

# Fixed point coordinates (LSE)
FIXED_LAT = 51.514114
FIXED_LON = -0.116971

most_merged['distance_to_lse'] = haversine_np(most_merged['long'],most_merged['lat'],FIXED_LON,FIXED_LAT)
most_merged.head(3)

,pcds,oslaua,lsoa11,imd,lat,long,duration_to_central,total_walking,borough,distance_to_lse
0,E1 7AA,E09000001,E01000005,8678,51.515567,-0.075635,30,20,City of London,2.868134
1,IG11 0AG,E09000002,E01000093,2669,51.531241,0.106421,65,32,Barking and Dagenham,15.589932
2,NW11 9EH,E09000003,E01000221,2878,51.573205,-0.211092,49,23,Barnet,9.259049


In [17]:
def speedcalc(duration, distance):
    time_hours = duration/60
    average_speed = distance/time_hours
    return average_speed

most_merged["average_speed_kmh"] = speedcalc(most_merged["duration_to_central"], most_merged["distance_to_lse"])
most_merged.head()

,pcds,oslaua,lsoa11,imd,lat,long,duration_to_central,total_walking,borough,distance_to_lse,average_speed_kmh
0,E1 7AA,E09000001,E01000005,8678,51.515567,-0.075635,30,20,City of London,2.868134,5.736268
1,IG11 0AG,E09000002,E01000093,2669,51.531241,0.106421,65,32,Barking and Dagenham,15.589932,14.390707
2,NW11 9EH,E09000003,E01000221,2878,51.573205,-0.211092,49,23,Barnet,9.259049,11.337610
3,DA8 2AB,E09000004,E01000429,3591,51.478252,0.182770,71,21,Bexley,21.153301,17.876029
4,NW10 0AB,E09000005,E01000601,1192,51.553022,-0.253283,54,26,Brent,10.385458,11.539398


In [18]:
most_merged.to_csv("data/processed/most_deprived_processed.csv", index=False)

### Section 2.2: Least Deprived Areas

In [19]:
least_df = pd.read_csv("data/raw/least_deprived_postcodes.csv")
least_df = least_df[["pcds", "oslaua", "lsoa11", "imd", "lat", "long", "duration_to_central", "total_walking"]]
least_merged = least_df.merge(borough_names, left_on='oslaua', right_on='LAD25CD', how='left')
print(len(least_merged["LAD25NM"].unique()) == 33)

True


In [20]:
least_merged = least_merged.drop("LAD25CD", axis=1)
least_merged.rename(columns={"LAD25NM": "borough"}, inplace=True)
least_merged.head()

,pcds,oslaua,lsoa11,imd,lat,long,duration_to_central,total_walking,borough
0,EC1Y 4AG,E09000001,E01000002,30379,51.520686,-0.090178,28,10,City of London
1,IG11 9AA,E09000002,E01000069,17580,51.541338,0.098494,55,22,Barking and Dagenham
2,N20 8DN,E09000003,E01000280,31544,51.630455,-0.184129,58,23,Barnet
3,DA5 1DY,E09000004,E01000448,32132,51.440866,0.147384,65,14,Bexley
4,HA3 0PF,E09000005,E01000539,25260,51.577084,-0.301107,58,27,Brent


In [21]:
least_merged['distance_to_lse'] = haversine_np(least_merged['long'],least_merged['lat'],FIXED_LON,FIXED_LAT)
least_merged["average_speed_kmh"] = speedcalc(least_merged["duration_to_central"], least_merged["distance_to_lse"])
least_merged.head()

,pcds,oslaua,lsoa11,imd,lat,long,duration_to_central,total_walking,borough,distance_to_lse,average_speed_kmh
0,EC1Y 4AG,E09000001,E01000002,30379,51.520686,-0.090178,28,10,City of London,1.994978,4.274952
1,IG11 9AA,E09000002,E01000069,17580,51.541338,0.098494,55,22,Barking and Dagenham,15.226835,16.611093
2,N20 8DN,E09000003,E01000280,31544,51.630455,-0.184129,58,23,Barnet,13.759329,14.233789
3,DA5 1DY,E09000004,E01000448,32132,51.440866,0.147384,65,14,Bexley,20.060243,18.517148
4,HA3 0PF,E09000005,E01000539,25260,51.577084,-0.301107,58,27,Brent,14.547694,15.049339


In [22]:
least_merged.to_csv("data/processed/least_deprived_processed.csv", index=False)

**Personal Reflection Note:**

In this section, I did most of the same in terms of adding borough names to each of the postcode entries in my CSVs for both the most deprived and least deprived areas. I also added a function to calculate the Haversine distance between each postcode based on its latitude and longitude (which was included in the postcode directory CSV), based on adapted code from a [Stack Overflow thread Jon had linked on Slack](https://stackoverflow.com/questions/29545704/fast-haversine-approximation-python-pandas/29546836#29546836). I then also added a function to calculate the "average speed" of each journey, which would help provide another dimension to my duration-based measure of connectivity, using a simple speed = distance/time calculation, ensuring that my time was converted from the minutes that the TfL Journey Planner API outputs into the hours that the speed calculation requires. I debugged and troubleshooted my code for the functions [using Claude](https://claude.ai/share/31aeabb8-2cac-466c-8f6b-a1d608c22fcc), and also used it to get a bit of a "sanity check" to see if I was doing my speed calculation function correctly. I also chose to save my data for the most deprived and least deprived areas separately, as I had done in NB01, as I find this to be clearer and more intuitive to me than creating a combined CSV with a separate column (perhaps made using a boolean) to indicate which postcodes are most deprived and least deprived.